In [1]:
# Attention 연산을 구현해 보아요!
import tensorflow as tf

# 예제 Text
tokens = ["나는", "너를", "사랑해"]

# 숫자로 변경해야 해요(원래는 단어사전을 이용해서 해야 해요)
word_to_index = {word : i for i, word in enumerate(tokens)}
print(word_to_index)

2025-07-11 10:11:10.316711: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-11 10:11:10.448513: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-07-11 10:11:10.515158: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-07-11 10:11:10.515554: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-07-11 10:11:10.607586: I tensorflow/core/platform/cpu_feature_gua

{'나는': 0, '너를': 1, '사랑해': 2}


In [2]:
# 숫자의 시퀀스로 변경해야 해요 -> [0 1 2]
input_ids = [word_to_index[word] for word in tokens]
print(input_ids)

[0, 1, 2]


In [3]:
# 이제 입력데이터를 숫자로 표현했어요 이제 Embedding부터 처리해 보아요!
# Embedding처리는 Tensorflow keras의 Embedding Layer를 이용할거에요
# 압력데이터를 Tensorflow Tensor로 변경
input_tensor = tf.constant(input_ids)
print(input_tensor)

tf.Tensor([0 1 2], shape=(3,), dtype=int32)


2025-07-11 10:33:51.021977: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-07-11 10:33:51.153653: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-07-11 10:33:51.153717: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-07-11 10:33:51.159819: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-07-11 10:33:51.159888: I external/local_xla/xla/stream_executor

In [5]:
# Embedding Layer
embedding_dim = 4
embedding_layer = tf.keras.layers.Embedding(input_dim=3, #input_dim=은 단어사전수(토큰 수)
                                           output_dim=embedding_dim)
embeddings = embedding_layer(input_tensor)
print(embeddings.shape)  #(3, 4)
print(embeddings.numpy())  # embedding 값

(3, 4)
[[ 0.01732676  0.04731591 -0.02396016  0.01664039]
 [-0.02496516  0.02011036 -0.00293661 -0.00090361]
 [ 0.00781429 -0.04870074 -0.03242301  0.02177214]]


In [7]:
# 쿼리. 키, 밸류르 추출하기 위해서 댄스 레이어가 3개 필요
d_model = 4
W_Q = tf.keras.layers.Dense(units=d_model, use_bias=False)
W_K = tf.keras.layers.Dense(units=d_model, use_bias=False)
W_V = tf.keras.layers.Dense(units=d_model, use_bias=False)

Q = W_Q(embeddings)
K = W_K(embeddings)
V = W_V(embeddings)

print("생성된 Query값")
print(Q.numpy())

print("생성된 Key값")
print(K.numpy())

생성된 Query값
[[-0.01746702 -0.0442015  -0.01122629  0.04075373]
 [ 0.01002997  0.00947697 -0.01788738  0.00559577]
 [-0.00970518  0.01520338 -0.00901021 -0.00858446]]
생성된 Key값
[[-0.0580092  -0.00749069  0.03178846  0.00235746]
 [ 0.00920283  0.00207186  0.01171805 -0.00810642]
 [-0.0252298  -0.03854601 -0.00371786  0.00690235]]


In [16]:
# 여기까지 수행되었으면 우리 문자열 (나는 너를 사랑해)에 대해
# 쿼리, 키 ,밸류를 구해날 수 있어요
# Attention score를 구할 수 있어요
d_k = tf.cast(tf.shape(K)[-1], tf.float32)
print(d_k)

scores = tf.matmul(Q, K, transpose_b=True) / tf.math.sqrt(d_k)
print(scores.shape) # (3, 3)

attention_weight = tf.nn.softmax(scores, axis=-1)  # 가장 마지막의 행
print(attention_weight.shape) # (3, 3)
print(attention_weight)
# 확률값으로 가중치를 구할 수 있어요!

output = tf.matmul(attention_weight, V) # (3, 4)
print(output.numpy())

tf.Tensor(4.0, shape=(), dtype=float32)
(3, 3)
(3, 3)
tf.Tensor(
[[0.33335626 0.33305675 0.33358702]
 [0.33323553 0.33341306 0.33335137]
 [0.33337477 0.3333354  0.33328986]], shape=(3, 3), dtype=float32)
[[-7.4446322e-03  8.7136356e-03 -7.7307224e-05 -1.9389978e-03]
 [-7.4392240e-03  8.7269768e-03 -6.3609332e-05 -1.9243238e-03]
 [-7.4359775e-03  8.7305270e-03 -5.7829544e-05 -1.9204756e-03]]
